In [3]:
import pandas as pd
import numpy as np

# ===================================================================
# FILE 1 SAVE: Raw Merging (Dirty Baseline Dataset)
# ===================================================================
sales_pipeline = pd.read_csv('/content/sales_pipeline.csv')
accounts = pd.read_csv('/content/accounts.csv')
products = pd.read_csv('/content/products.csv')
sales_teams = pd.read_csv('/content/sales_teams.csv')

# 4 Raw tables ko join karke dirty master dataframe banayein
df = sales_pipeline.merge(accounts, on='account', how='left')
df = df.merge(products, on='product', how='left')
df = df.merge(sales_teams, on='sales_agent', how='left')

# Save File 1 (Dirty Baseline)
df.to_csv('crm_sales_dirty_master.csv', index=False)
print("✅ FILE 1 SAVED: 'crm_sales_dirty_master.csv' (Raw Merged Baseline)")

✅ FILE 1 SAVED: 'crm_sales_dirty_master.csv' (Raw Merged Baseline)


In [8]:
# ===================================================================
# DATA CLEANING & TRANSFORMATION STEPS
# ===================================================================
# 1. Text & Typo Fixes
df['product'] = df['product'].replace({'GTXPro': 'GTX Pro'})
df['sector'] = df['sector'].replace({'technolgy': 'technology'})

# 2. Drop Redundant Columns
cols_to_drop = ['subsidiary_of', 'series', 'sales_price', 'year_established']
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 3. Date Modernization (+7 Years Shift to 2023-2025)
engage_dt = pd.to_datetime(df['engage_date']) + pd.DateOffset(years=7)
close_dt = pd.to_datetime(df['close_date']) + pd.DateOffset(years=7)

df['engage_date'] = engage_dt.dt.strftime('%Y-%m-%d')
df['close_date'] = close_dt.dt.strftime('%Y-%m-%d')

# 4. Null Imputation
df['account'] = df['account'].fillna('Unassigned Account')
df['sector'] = df['sector'].fillna('Unassigned Sector')
df['office_location'] = df['office_location'].fillna('Unknown Location')

df['revenue'] = df.groupby('sector')['revenue'].transform(lambda x: x.fillna(x.median()))
df['employees'] = df.groupby('sector')['employees'].transform(lambda x: x.fillna(x.median()))
df['revenue'] = df['revenue'].fillna(df['revenue'].median())
df['employees'] = df['employees'].fillna(df['employees'].median())

# 5. Outliers Handling (Capping Extreme Values at 99th Percentile)
cap_close_val = df['close_value'].quantile(0.99)
df['close_value_capped'] = np.where(df['close_value'] > cap_close_val, cap_close_val, df['close_value'])

cap_revenue = df['revenue'].quantile(0.99)
df['revenue_capped'] = np.where(df['revenue'] > cap_revenue, cap_revenue, df['revenue'])

# 6. Feature Engineering
df['is_won'] = np.where(df['deal_stage'] == 'Won', 1, 0)
df['sales_cycle_days'] = (close_dt - engage_dt).dt.days
df['close_year_quarter'] = close_dt.dt.to_period('Q').astype(str)
df['close_year_quarter'] = df['close_year_quarter'].replace('NaT', np.nan)
df['is_closed'] = np.where(df['deal_stage'].isin(['Won', 'Lost']), 1, 0)


# ===================================================================
# FILE 2 SAVE: Cleaned Master Dataset (Final Output)
# ===================================================================
df.to_csv('crm_sales_cleaned_master.csv', index=False)
print("✅ Fully Cleaned Dataset with Outliers Handled Saved!")

✅ Fully Cleaned Dataset with Outliers Handled Saved!


(8800, 20)